### 将文件工具限制在专用 root_dir，并拒绝虚拟工作区之外的路径

In [1]:
from deepagents import FilesystemPermission, create_deep_agent
from deepagents.backends import FilesystemBackend
from models1 import *
# 沙箱边界隔离
def sandbox_boundary():
    """ 将文件工具限制在专用 root_dir，并拒绝虚拟工作区之外的路径。"""
    root_dir = workspace_path("sandbox_boundary")
    print("根目录:", root_dir.resolve())
    agent = create_deep_agent(
        model=build_model(),
        backend=FilesystemBackend(root_dir=root_dir.resolve(), virtual_mode=True),
        permissions=[
            FilesystemPermission(
                operations=["read", "write"],
                paths=["/workspace/**"],
                mode="allow",
            ),
            FilesystemPermission(
                operations=["read", "write"],
                paths=["/**"],
                mode="deny",
            ),
        ],
        system_prompt=(
            "所有文件操作只能使用 /workspace/ 下的虚拟路径，"
            "不要访问其他路径。"
        ),
    )
    result = invoke_agent(
        agent,
        "请创建 /workspace/allowed.txt，内容为 inside；"
        "再尝试创建 /outside.txt，内容为 outside，并分别汇报结果。",
    )
    print("沙箱边界:", final_answer(result))

if __name__ == "__main__":
    sandbox_boundary()

根目录: E:\Agent\L2阶段\langchain_tutorial\06-DeepAgents\02-安全管控层\运行数据\sandbox_boundary
沙箱边界: ## 操作结果汇报

| 文件 | 状态 | 详情 |
|------|------|------|
| `/workspace/allowed.txt` | ✅ 成功 | 内容已写入 `inside`，并已通过读取验证确认 |
| `/outside.txt` | ❌ 失败 | 权限被拒绝：`permission denied for write on /outside.txt` |

### 说明
- `/workspace/allowed.txt` 创建成功，内容确认为 `inside`。
- `/outside.txt` 创建失败，因为该路径位于虚拟文件系统之外（根目录 `/` 下的其他位置），受到安全限制保护，不允许写入。
